Import the dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("CADCODER/GenCAD-Code")

train = ds['train']
test = ds['test']
eval = ds['validation']

In [ ]:
import random
random.seed(42)
eval_subset = eval.shuffle(seed=42).select(range(200))

Download the checkpoint

In [ ]:
from huggingface_hub import create_repo
import os

HUB_REPO = ""
os.environ["HF_TOKEN"] = ""
checkpoint = "sft"

create_repo(repo_id=HUB_REPO, private=True, exist_ok=True)

local_dir = "checkpoint_local_path"

if checkpoint == "sft" :
    from gdrive_fsspec import GoogleDriveFileSystem

    fs = GoogleDriveFileSystem(use_listings_cache=False, skip_instance_cache=True, 
                               auth_kwargs={"use_local_webserver": False})
    fs.get("drive_file_path", local_dir, recursive=True)


else:
    from huggingface_hub import hf_hub_download, snapshot_download

    # revision = ""
    # path = snapshot_download(repo_id = HUB_REPO, revision = revision, local_dir = local_dir)
    path = snapshot_download(repo_id= HUB_REPO,allow_patterns=f"{checkpoint}/*", local_dir=local_dir)
    print(path)

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

Load the model

In [ ]:
import accelerate
from transformers import AutoProcessor, AutoModelForMultimodalLM
from peft import PeftModel

processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-4B")
base_model = AutoModelForMultimodalLM.from_pretrained("Qwen/Qwen3.5-4B", device_map="auto")

model = PeftModel.from_pretrained(base_model, local_dir)
# model = base_model
model.eval()

To extract only the python code and nothing else from what the model returns:

In [ ]:
import re

_CODE_FENCE_RE = re.compile(r'```(?:python)?\s*\n?(.*?)\n?```', re.DOTALL)

def extract_code(text):
    match = _CODE_FENCE_RE.search(text)
    if match:
        return match.group(1).strip()
    return text.strip()

Compute IOUs using best_iou

In [ ]:
from best_iou import get_iou_best
from tqdm import tqdm

results = []
batch_size = 32

model.eval()

for batch_start in tqdm(range(0, len(eval_subset), batch_size)):
    batch = eval_subset[batch_start:batch_start + batch_size]
    batch_images = batch["image"]
    batch_texts = [
        processor.apply_chat_template(
            [{"role": "user",
        "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": "Generate the CadQuery Python code to create this 3D CAD model. Return only the code, no explanation." }]}],
            tokenize=False, add_generation_prompt=True,enable_thinking=False) for image in batch_images]

    inputs = processor(
        text=batch_texts,
        images=batch_images,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=4096,
            do_sample=False,
            repetition_penalty=1.3,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[:, prompt_len:]

    generated_texts = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )

    for deepcad_id, cadquery, generated_text in zip(
        batch["deepcad_id"],
        batch["cadquery"],
        generated_texts,
    ):
        extracted_text = extract_code(generated_text)

        try:
            iou = get_iou_best(cadquery, extracted_text)
            results.append({
                "id": deepcad_id,
                "generated_code": extracted_text,
                "generated_text" : generated_text,
                "iou": iou,
                "error": None,
            })
        except Exception as e:
            results.append({
                "id": deepcad_id,
                "generated_text" : generated_text,
                "generated_code": extracted_text,
                "iou": 0,
                "error": str(e),
            })

    del inputs, outputs, generated_ids

Check mean, median and standard deviation of IOUs

In [ ]:
import numpy as np
import json

def get_results(filename = None, results = None):
    if filename is not None:
        with open (filename, "r") as f:
            results = json.load(f)
    ious = [r["iou"] for r in results if r["error"] is None]
    print(f"Mean IoU:   {np.mean(ious):.4f}")
    print(f"Median IoU: {np.median(ious):.4f}")
    print(f"Std Dev:    {np.std(ious):.4f}")
    print(len(results))
    print(sum(r["error"] is None for r in results))
    print(ious[:10])

In [ ]:
get_results(results = results)

In [ ]:
filename = "results.json"
with open (filename, "w") as f:
    json.dump(results, f)

In [ ]:
qwen4b_14k = "results.json"
get_results(filename=qwen4b_14k)

Check loss curves using trainer_state.json

In [ ]:
import json
with open("trainer_state.json") as f:
    state = json.load(f)

for entry in state["log_history"]:
    print(entry.get("step"), entry.get("loss"), entry.get("eval_loss"))

In [ ]:
import matplotlib.pyplot as plt

steps = [entry["step"] for entry in state["log_history"] if "loss" in entry]
losses = [entry["loss"] for entry in state["log_history"] if "loss" in entry]
eval_steps = [entry["step"] for entry in state["log_history"] if "eval_loss" in entry]
eval_loss = [entry["eval_loss"] for entry in state["log_history"] if "eval_loss" in entry]

plt.plot(steps, losses)
plt.plot(eval_steps, eval_loss)
plt.xlabel("step")
plt.ylabel("loss")
plt.show()